### Composable LLM Application using LCEL

This project demonstrates how to build a simple yet extensible LLM application using LangChain Expression Language (LCEL).

The system is designed as a composable pipeline, where prompts, models, and output parsers are combined to perform different tasks such as translation, summarization, and explanation.

### Key Concepts Covered

- Using Large Language Models (Groq - Gemma)
- Prompt Templates and dynamic input handling
- Output parsing with `StrOutputParser`
- LCEL chaining (`prompt | model | parser`)
- Exposing LLM pipelines as APIs using FastAPI and LangServe

This project focuses on building a reusable and modular LLM pipeline rather than a single-purpose application.

LCEL (LangChain Expression Language) allows us to compose prompts, models, and parsers into reusable pipelines using the | operator.

In [2]:
###### Loading Groq API Key and Initializing a Chat Model

import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")


In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001BCBABE38B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BCBAC7FF10>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [5]:
result

AIMessage(content='Bonjour, comment ça va\u202f?', additional_kwargs={'reasoning_content': 'We need to translate "Hello How are you?" to French. Probably "Bonjour, comment ça va ?" or "Bonjour, comment allez-vous ?" depending. The user said "Hello How are you?" likely informal. So "Bonjour, comment ça va ?" Provide translation.'}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 86, 'total_tokens': 157, 'completion_time': 0.150231126, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.003541105, 'prompt_tokens_details': None, 'queue_time': 0.045153995, 'total_time': 0.153772231}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d25f8-ee44-7e71-aee7-385159e2d31d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 71, 'total_tokens': 157, 'output_token_detail

In [6]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment ça va\u202f?'

In [7]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour, comment ça va\u202f?'

In [8]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [9]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [10]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [11]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour'

In [12]:
chain.invoke({"language": "Telugu", "text": "Good morning"})

'శుభోదయం (Good morning)'